In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
 
# Налаштування стилю
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
FIGSIZE = (14, 6)
 
# Завантаження даних
df = pd.read_csv('facebook_ads_data (2.0).csv')
df['ad_date'] = pd.to_datetime(df['ad_date'])
 
print("Дані завантажено:", df.shape)
print(df.head(3))

✅ Дані завантажено: (1494, 10)
     ad_date campaign_name  total_spend  total_impressions  total_clicks  \
0 2022-11-05     Expansion         0.00                  0             0   
1 2022-11-01     Expansion         0.00                  0             0   
2 2022-10-31     Expansion       227.45               6054            58   

   total_value   cpc    cpm      ctr     romi  
0         0.00   NaN    NaN      NaN      NaN  
1         0.00   NaN    NaN      NaN      NaN  
2       191.87  3.92  37.57  0.00958  0.84357  


In [4]:
# === ЗАВДАННЯ 1 ===
 
df_2021 = df[(df['ad_date'] >= '2021-01-01') & (df['ad_date'] < '2022-01-01')]
 
daily = (
    df_2021.groupby('ad_date')
    .agg(total_spend=('total_spend', 'sum'),
         total_value=('total_value', 'sum'))
    .reset_index()
)
daily['romi'] = daily['total_value'] / daily['total_spend'].replace(0, float('nan'))
daily = daily.sort_values('ad_date')
 
# Рухоме середнє (7 днів)
daily['spend_rolling'] = daily['total_spend'].rolling(7, min_periods=1).mean()
daily['romi_rolling']  = daily['romi'].rolling(7, min_periods=1).mean()
 
fig, axes = plt.subplots(2, 1, figsize=(16, 10))
fig.suptitle('Щоденні показники реклами у 2021 році', fontsize=16, fontweight='bold', y=1.01)
 
# --- Графік 'Щоденні витрати' ---
ax = axes[0]
ax.bar(daily['ad_date'], daily['total_spend'], color='steelblue', alpha=0.5, label='Витрати (щоденно)')
ax.plot(daily['ad_date'], daily['spend_rolling'], color='darkblue', linewidth=2, label='Рухоме середнє (7 днів)')
ax.set_title('Щоденна сума витрат на рекламу', fontsize=13)
ax.set_xlabel('Дата')
ax.set_ylabel('Витрати ($)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.legend()
ax.tick_params(axis='x', rotation=45)
 
# --- Графік 'Щоденний ROMI' ---
ax = axes[1]
ax.bar(daily['ad_date'], daily['romi'], color='mediumseagreen', alpha=0.5, label='ROMI (щоденно)')
ax.plot(daily['ad_date'], daily['romi_rolling'], color='darkgreen', linewidth=2, label='Рухоме середнє (7 днів)')
ax.axhline(1, color='red', linestyle='--', alpha=0.7, label='ROMI = 1 (беззбитковість)')
ax.set_title('Щоденний ROMI (Return on Marketing Investment)', fontsize=13)
ax.set_xlabel('Дата')
ax.set_ylabel('ROMI')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.legend()
ax.tick_params(axis='x', rotation=45)
 
plt.tight_layout()
plt.savefig('task1_daily_2021.png', dpi=150, bbox_inches='tight')
plt.close()
print("Завдання 1 збережено: task1_daily_2021.png")

✅ Завдання 1 збережено: task1_daily_2021.png


In [5]:
# === ЗАВДАННЯ 2 ===

campaign = (
    df.groupby('campaign_name')
    .agg(total_spend=('total_spend', 'sum'),
         total_value=('total_value', 'sum'))
    .reset_index()
)
campaign['romi'] = campaign['total_value'] / campaign['total_spend'].replace(0, float('nan'))
campaign = campaign.sort_values('total_spend', ascending=False)
 
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle('Показники по кампаніях (всі роки)', fontsize=15, fontweight='bold')
 
# Графік 'Загальні витрати по кампаніях' ---
bars = axes[0].barh(campaign['campaign_name'], campaign['total_spend'],
                    color=sns.color_palette("husl", len(campaign)))
axes[0].set_title('Загальна сума витрат на рекламу по кампаніях', fontsize=12)
axes[0].set_xlabel('Загальні витрати ($)')
axes[0].set_ylabel('Кампанія')
for bar, val in zip(bars, campaign['total_spend']):
    axes[0].text(bar.get_width() + 200, bar.get_y() + bar.get_height()/2,
                 f'${val:,.0f}', va='center', fontsize=9)
 
# Графік 'Загальний ROMI по 'кампаніях' ---
campaign_romi = campaign.sort_values('romi', ascending=False)
colors = ['#2ecc71' if r >= 1 else '#e74c3c' for r in campaign_romi['romi']]
bars2 = axes[1].barh(campaign_romi['campaign_name'], campaign_romi['romi'], color=colors)
axes[1].axvline(1, color='black', linestyle='--', alpha=0.6, label='ROMI = 1')
axes[1].set_title('Загальний ROMI по кампаніях', fontsize=12)
axes[1].set_xlabel('ROMI')
axes[1].set_ylabel('Кампанія')
axes[1].legend()
for bar, val in zip(bars2, campaign_romi['romi']):
    axes[1].text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2,
                 f'{val:.2f}', va='center', fontsize=9)
 
plt.tight_layout()
plt.savefig('task2_campaigns.png', dpi=150, bbox_inches='tight')
plt.close()
print("Завдання 2 збережено: task2_campaigns.png")

Завдання 2 збережено: task2_campaigns.png


In [6]:
# === ЗАВДАННЯ 3 ===
df_romi = df.dropna(subset=['romi'])
 
fig, ax = plt.subplots(figsize=(16, 7))
campaign_order = (df_romi.groupby('campaign_name')['romi']
                  .median().sort_values(ascending=False).index.tolist())
 
sns.boxplot(data=df_romi, x='campaign_name', y='romi', order=campaign_order,
            palette='husl', ax=ax, showfliers=True,
            flierprops=dict(marker='o', markerfacecolor='gray', alpha=0.4, markersize=4))
ax.axhline(1, color='red', linestyle='--', alpha=0.8, label='ROMI = 1')
ax.set_title('Розкид щоденного ROMI по кампаніях (Box Plot)', fontsize=14, fontweight='bold')
ax.set_xlabel('Кампанія', fontsize=12)
ax.set_ylabel('ROMI', fontsize=12)
ax.tick_params(axis='x', rotation=30)
ax.legend(fontsize=11)
 
plt.tight_layout()
plt.savefig('task3_boxplot_romi.png', dpi=150, bbox_inches='tight')
plt.close()
print("Завдання 3 збережено: task3_boxplot_romi.png")

Завдання 3 збережено: task3_boxplot_romi.png


In [7]:
# === ЗАВДАННЯ 4 ===

fig, ax = plt.subplots(figsize=(12, 6))
romi_clean = df['romi'].dropna()
romi_clip = romi_clean.clip(lower=romi_clean.quantile(0.01), upper=romi_clean.quantile(0.99))
 
ax.hist(romi_clip, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
ax.axvline(romi_clip.mean(), color='red', linestyle='--', linewidth=2,
           label=f'Середнє = {romi_clip.mean():.2f}')
ax.axvline(romi_clip.median(), color='orange', linestyle='--', linewidth=2,
           label=f'Медіана = {romi_clip.median():.2f}')
ax.axvline(1, color='green', linestyle='-', linewidth=1.5, alpha=0.8, label='ROMI = 1')
ax.set_title('Розподіл значень ROMI (facebook_ads_data)', fontsize=14, fontweight='bold')
ax.set_xlabel('ROMI', fontsize=12)
ax.set_ylabel('Кількість спостережень', fontsize=12)
ax.legend(fontsize=11)
 
plt.tight_layout()
plt.savefig('task4_romi_histogram.png', dpi=150, bbox_inches='tight')
plt.close()
print("Завдання 4 збережено: task4_romi_histogram.png")

Завдання 4 збережено: task4_romi_histogram.png


In [8]:
# === ЗАВДАННЯ 5 ===

numeric_cols = df.select_dtypes(include='number').columns.tolist()
corr = df[numeric_cols].corr()
 
fig, ax = plt.subplots(figsize=(11, 9))
mask = pd.DataFrame(False, index=corr.index, columns=corr.columns)
import numpy as np
mask_tri = np.triu(np.ones_like(corr, dtype=bool), k=1)
 
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            linewidths=0.5, ax=ax, square=True,
            cbar_kws={'shrink': 0.8}, vmin=-1, vmax=1)
ax.set_title('Теплова карта кореляції між числовими показниками', fontsize=14, fontweight='bold')
plt.xticks(rotation=30, ha='right')
plt.yticks(rotation=0)
 
plt.tight_layout()
plt.savefig('task5_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.close()
 
# Аналіз кореляцій
corr_flat = corr.unstack()
corr_flat = corr_flat[corr_flat < 1].sort_values()
print("Найнижча кореляція:")
print(corr_flat.head(3))
print("Найвища кореляція:")
print(corr_flat.tail(3))
print("Кореляція total_value з іншими:")
print(corr['total_value'].sort_values(ascending=False))
print("Завдання 5 збережено: task5_correlation_heatmap.png")

Найнижча кореляція:
ctr           cpc   -0.210719
cpc           ctr   -0.210719
total_clicks  cpc   -0.159521
dtype: float64
Найвища кореляція:
total_impressions  total_clicks    0.765489
total_value        total_spend     0.978890
total_spend        total_value     0.978890
dtype: float64
Кореляція total_value з іншими:
total_value          1.000000
total_spend          0.978890
total_clicks         0.472124
total_impressions    0.472037
cpm                  0.471338
cpc                  0.250851
romi                -0.013733
ctr                 -0.022267
Name: total_value, dtype: float64
Завдання 5 збережено: task5_correlation_heatmap.png


In [9]:
# === ЗАВДАННЯ 6 ===
 
df_clean = df.dropna(subset=['total_spend', 'total_value'])
df_clean = df_clean[df_clean['total_spend'] > 0]
 
g = sns.lmplot(
    data=df_clean,
    x='total_spend',
    y='total_value',
    height=7,
    aspect=1.6,
    scatter_kws={'alpha': 0.35, 's': 30, 'color': 'steelblue'},
    line_kws={'color': 'red', 'linewidth': 2.5}
)
 
r = df_clean['total_spend'].corr(df_clean['total_value'])
g.ax.set_title('Залежність між витратами та цінністю реклами\n(total_spend vs total_value)',
               fontsize=14, fontweight='bold')
g.ax.set_xlabel('Загальні витрати (total_spend, $)', fontsize=12)
g.ax.set_ylabel('Загальна цінність (total_value, $)', fontsize=12)
g.ax.text(0.05, 0.92, f'r = {r:.3f}', transform=g.ax.transAxes,
          fontsize=13, color='darkred',
          bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.8))
 
plt.savefig('task6_regression.png', dpi=150, bbox_inches='tight')
plt.close()
print("Завдання 6 збережено: task6_regression.png")

Завдання 6 збережено: task6_regression.png
